[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C30_Agent_Harness_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 Python 标准库、CPU 可跑、无需 API key**：用一个返回预设动作的 **MockLLM**（假大脑）驱动 agent，让每个 agent 都**端到端真实运行**并用 `assert` 验证；同时每个 notebook 都附「换上真实 Claude API」的适配代码（**有 key 接真模型、没 key 自动回退 MockLLM**）。

这个 notebook 做三件事：① 确认环境（只需标准库）；② 用一个**最小 agent 循环**亲手体会「agent = 模型 + 循环 + 工具 + 上下文」；③ 立下全课的纪律——**用确定性 MockLLM + assert 验证机制**。

## 1 · 环境自检

本课**不需要任何第三方包**就能学完全部机制（MockLLM 路径）。只有想接真实 Claude 时才需要 `anthropic`。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
assert sys.version_info >= (3, 8), '建议 Python 3.8+'

# 真实 Claude 适配是可选的：有就用，没有也不影响学习
try:
    import anthropic
    print('anthropic SDK', anthropic.__version__, '(可选，用于接真实 Claude)')
except Exception:
    print('anthropic 未安装（可选）—— 本课用 MockLLM 即可端到端跑通')

import os
has_key = bool(os.environ.get('ANTHROPIC_API_KEY'))
print('ANTHROPIC_API_KEY', '已设置（可接真实 Claude）' if has_key else '未设置（走 MockLLM 回退）')
print('环境就绪 ✅  —— 无论有没有 key，下面的代码都能跑')

## 2 · 一个最小 agent 循环：30 行看清「自主」的真相

agent 的「自主」没有魔法，就是 **感知→决策→行动→观察** 在转。下面用一个**极简但完整**的循环，配一个 MockLLM 假大脑，让你亲眼看到一个 agent 多步完成任务。先建一个会按脚本走的 MockLLM。

In [ ]:
class MockLLM:
    '''假大脑：按预设脚本逐步返回「下一步动作」。确定性、不联网、零成本。
       每个动作是一个 dict：
         {'type':'tool', 'name':..., 'input':...}  -> 要调用工具
         {'type':'final', 'text':...}              -> 给出最终答案
    '''
    def __init__(self, script):
        self.script = list(script)
        self.calls = 0
    def decide(self, history):
        # 真实 LLM 会读 history 再决定；MockLLM 直接播放下一条脚本
        action = self.script[self.calls]
        self.calls += 1
        return action

# 脚本：先查天气、再根据结果给答案
brain = MockLLM(script=[
    {'type': 'tool',  'name': 'get_weather', 'input': {'city': '北京'}},
    {'type': 'final', 'text': '北京今天晴，22°C，适合出门。'},
])
print('MockLLM 就绪，脚本有', len(brain.script), '步')

现在写**循环本身**（agent 的本体），再给它一个工具，跑起来。注意循环里清晰的四个阶段。

In [ ]:
def get_weather(city):
    # 一个玩具工具：真实里会调天气 API，这里返回固定值
    return f'{city}: 晴, 22°C'

TOOLS = {'get_weather': get_weather}

def run_agent(brain, tools, task, max_steps=10):
    history = [{'role': 'user', 'content': task}]   # 上下文/scratchpad
    trace = []                                      # 记录轨迹便于验证
    for step in range(max_steps):                   # max-steps 防护！
        action = brain.decide(history)              # ② 决策
        if action['type'] == 'final':               # ③ 行动:是最终答案?
            trace.append(('final', action['text']))
            return action['text'], trace            #    -> 停止
        # 否则是工具调用 -> 执行
        name, args = action['name'], action['input']
        result = tools[name](**args)                # ③ 行动:执行工具
        trace.append(('tool', name, result))
        history.append({'role': 'observation',      # ④ 观察:回灌
                        'content': f'{name} -> {result}'})
    raise RuntimeError('超过 max_steps 仍未结束（疑似死循环）')

answer, trace = run_agent(brain, TOOLS, '北京今天天气怎样？')
print('最终答案:', answer)
print('轨迹:')
for t in trace:
    print('  ', t)

**用 assert 验证这个 agent 确实按预期多步完成了任务**——这是本课的核心纪律：因为 MockLLM 确定性，轨迹完全可断言。

In [ ]:
# 因为 MockLLM 是确定性的，我们能精确断言 agent 的行为
assert trace[0][0] == 'tool' and trace[0][1] == 'get_weather', '第 1 步应调用 get_weather'
assert trace[0][2] == '北京: 晴, 22°C', '工具应返回正确观察'
assert trace[1][0] == 'final', '第 2 步应给出最终答案'
assert '北京' in answer
assert len(trace) == 2, '整个任务恰好 2 步完成'
assert brain.calls == 2, 'MockLLM 恰好被问了 2 次'
print('✅ agent 端到端跑通：查天气 -> 据结果作答，2 步完成，全部断言通过')
print('这就是本课的工作流：写 harness -> 用 MockLLM 端到端跑 -> assert 兜底验证机制')

## 3 · 为什么需要 max-steps：亲眼看一个死循环被拦住

如果模型永远不给最终答案（真实里可能因为陷入困惑、或被对抗输入诱导），没有 `max_steps` 的 agent 会**永远转下去**。下面用一个「永远只调工具、从不收尾」的坏脚本，验证防护确实生效。

In [ ]:
# 一个永不收尾的坏大脑：每步都重复调同一个工具
bad_brain = MockLLM(script=[{'type':'tool','name':'get_weather','input':{'city':'北京'}}] * 100)

looped = False
try:
    run_agent(bad_brain, TOOLS, '会死循环的任务', max_steps=5)
except RuntimeError as e:
    looped = True
    print('被拦住:', e)

assert looped, 'max_steps 必须拦住死循环'
assert bad_brain.calls == 5, '恰好在第 5 步触发上限'
print('✅ max_steps 守卫生效：永不收尾的 agent 在 5 步处被安全中止（模块 01/04 深入）')

## 4 · 真实 Claude 适配预览：无 key 自动回退

本课每个 notebook 都附「换上真实 Claude」的适配代码。核心姿态是一个 `make_llm()`：**有 `ANTHROPIC_API_KEY` 就接真实 `claude-opus-4-8`，没有就回退 MockLLM**。下面是这个姿态的骨架（真实分支需要 `anthropic` 与 key，这里以可跑的方式演示「选哪条路」的逻辑）。

In [ ]:
import os

def make_llm(mock_script):
    '''有 key -> 真实 Claude；无 key -> MockLLM。绝不因缺 key 阻断 notebook。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic   # 仅在真要用时才依赖
            # 真实实现见模块 03；这里只表达「会走真实路径」
            return ('real', 'claude-opus-4-8')
        except ImportError:
            pass                # 装了 key 但没装 SDK -> 也回退
    return ('mock', MockLLM(script=mock_script))

kind, llm = make_llm([{'type':'final','text':'hi'}])
print('本机将使用:', kind, '路径')
assert kind in ('real', 'mock')
# 无论哪条路径，agent 循环的代码完全一样（可插拔的力量）
print('✅ 适配姿态成立：同一份 agent 代码，有 key 接真模型、没 key 走 mock，都能跑')

## 5 · 全课地图：四块零件如何拼成一个 agent

本课把一个能跑的 agent 拆成四块，逐块从零造，最后拼起来。这张依赖链请记住——后面每个模块都在打其中一块。

In [ ]:
AGENT_PARTS = [
    # (模块, 零件, 解决什么)
    ('01', 'agent 循环',  '让模型多步行动：感知-决策-行动-观察 + 停止条件'),
    ('02', '工具系统',    '给模型手脚：注册/schema/分发/并行/错误隔离'),
    ('03', 'LLM 适配器',  '统一大脑接口：MockLLM <-> 真实 Claude 一行互换'),
    ('04', '鲁棒层',      '出错不崩：重试/超时/循环检测/成本追踪'),
    ('05', '最小 agent',  '把以上拼成能用工具多步完成任务的整体'),
]
print('依赖链：05 = (01 循环 + 02 工具) 跑在 (03 接口) 上，外裹 (04 鲁棒)')
for m, part, why in AGENT_PARTS:
    print(f'  模块 {m}: {part:10s} -> {why}')
# 验证：最小 agent 依赖前四块
deps = {p[1] for p in AGENT_PARTS[:4]}
assert deps == {'agent 循环', '工具系统', 'LLM 适配器', '鲁棒层'}
print('\n✅ 地图清晰。下一站：模块 01 · Agent 循环 —— 把这 30 行循环做扎实、做对。')

### 小结
- **agent ≠ 模型**：agent = 模型 + **循环** + **工具** + **上下文**，是围绕模型的一圈编排代码（harness）。
- **循环**：感知→决策→行动→观察反复，把无状态的 LLM 变成有状态的多步过程；状态藏在不断累积的 history 里。
- **MockLLM**：确定性假大脑，让 agent **端到端真实可跑 + assert 可验证 + 零成本**——本课所有测试的基石。
- **真实适配**：每个 notebook 附 Claude 适配代码，**有 key 接真模型、没 key 自动回退 MockLLM**，绝不阻断。
- **max-steps** 是 agent 最基本的安全阀（模块 01/04 深入）。

下一站：**模块 01 · Agent 循环** —— 把这圈循环做对、做稳，处理好 ReAct 解析与停止条件。